In [1]:
import pandas as pd
import numpy as np

# 1. Load the raw dataset
print("Loading raw dataset...")
df = pd.read_csv("CIC-TON-IoT.csv") 

# Replace infinity values with NaN, then drop all rows containing NaN
df.replace([np.inf, -np.inf], np.nan, inplace=True)
df.dropna(inplace=True)
# ------------------------------------------------------

# 2. Define the columns to keep
columns_to_keep = [
    'Src IP', 'Dst IP', 'Timestamp', 'Label', 
    'Src Port', 'Dst Port', 'Fwd Header Len', 'Init Bwd Win Byts', 
    'Fwd Seg Size Avg', 'Fwd Pkt Len Mean', 'Init Fwd Win Byts', 
    'Fwd Pkt Len Max', 'TotLen Fwd Pkts', 'Bwd Pkt Len Mean', 
    'Idle Min', 'Bwd Header Len', 'Pkt Len Var', 'Subflow Fwd Byts', 
    'TotLen Bwd Pkts', 'Idle Max', 'Fwd Seg Size Min', 'Idle Mean', 
    'Pkt Len Max', 'Bwd Pkt Len Std', 'Bwd Pkt Len Max', 
    'Protocol', 'Pkt Len Mean', 'Down/Up Ratio'
]

# Ensure column names match exactly
df.columns = df.columns.str.strip()
df = df[columns_to_keep]

# 3. Temporal Ordering
print("Sorting by Timestamp...")
# Note: CIC datasets sometimes have mixed date formats. 
# Added 'format='mixed'' (for pandas >= 2.0) or 'infer_datetime_format=True' for safety
df['Timestamp'] = pd.to_datetime(df['Timestamp'], errors='coerce')
df.dropna(subset=['Timestamp'], inplace=True) # Drop rows where time parsing failed
df = df.sort_values(by='Timestamp', ascending=True)

# 4. Standardize Labels (Ensure Benign is 0, Attack is 1)
df['Label'] = df['Label'].apply(lambda x: 0 if x in ['Normal', 'Benign', 0, '0'] else 1)

# 5. Split the Data
print("Splitting into Train (Benign only) and Test (Mixed)...")
benign_data = df[df['Label'] == 0]
attack_data = df[df['Label'] == 1]

# Take the first portion of the benign data for training
train_size = 100000 
train_data = benign_data.head(train_size)

# Take a different portion for testing, mixing remaining benign data with attacks
test_benign = benign_data.iloc[train_size : train_size + 50000] 
test_attacks = attack_data.head(50000)

test_data = pd.concat([test_benign, test_attacks])
test_data = test_data.sort_values(by='Timestamp', ascending=True)

# 6. Save to CSV
print("Saving train_data.csv and test_data.csv...")
train_data.to_csv("train_data.csv", index=False)
test_data.to_csv("test_data.csv", index=False)

print("Preprocessing complete! Ready for 'train for 3edge.ipynb'.")

Loading raw dataset...
Sorting by Timestamp...


/var/folders/vw/nm4pm2012rj1j5k7yc6x23bw0000gn/T/ipykernel_18388/4094595341.py:34: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['Timestamp'] = pd.to_datetime(df['Timestamp'], errors='coerce')


Splitting into Train (Benign only) and Test (Mixed)...
Saving train_data.csv and test_data.csv...
Preprocessing complete! Ready for 'train for 3edge.ipynb'.
